In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
movie=pd.read_csv("tmdb_5000_movies.csv")

In [3]:
credit=pd.read_csv("tmdb_5000_credits.csv")

In [4]:
dataset = movie.merge(credit,on="title")
pd.set_option("display.max_columns",None) # for viewing max coloumns

In [5]:
#dataset

In [6]:
dataset = dataset[["genres","id","overview","title","keywords","cast","crew"]]

In [7]:
dataset.dropna(inplace=True)

In [8]:
import ast # use for removing string on list " ' [ example ] ' "

def convert(obj):
    l = []
    for i in ast.literal_eval(obj):
        l.append(i["name"])
    return l

In [9]:
dataset["genres"] = dataset["genres"].apply(convert)

In [10]:
dataset["keywords"] = dataset["keywords"].apply(convert)

In [11]:
def convert2(obj):
    l2 = []
    counter=0
    for i in ast.literal_eval(obj):
        if counter != 0:
            l2.append(i["name"])
            counter+=1
        else:
            break

    return l2

In [12]:
dataset["cast"] = dataset["cast"].apply(convert)

In [13]:
def director(obj):
    l3 = []
    for i in ast.literal_eval(obj):
        if i["job"] == "Director":
            l3.append(i["name"])
            break

    return l3

In [14]:
dataset["crew"] = dataset["crew"].apply(director)

In [15]:
dataset["overview"] = dataset["overview"].apply(lambda x:x.split())

In [16]:
dataset["genres"] = dataset["genres"].apply(lambda x:[i.replace(" ","") for i in x])
dataset["keywords"] = dataset["keywords"].apply(lambda x:[i.replace(" ","") for i in x])
dataset["cast"] = dataset["cast"].apply(lambda x:[i.replace(" ","") for i in x])
dataset["crew"] = dataset["crew"].apply(lambda x:[i.replace(" ","") for i in x])

In [17]:
dataset["tags"] = dataset["overview"] + dataset["genres"] + dataset["keywords"] + dataset["cast"] + dataset["crew"]

In [18]:
dataset = dataset[["title","tags","id"]]

In [19]:
dataset["tags"] = dataset["tags"].apply(lambda x:" ".join(x))

In [20]:
dataset["tags"] = dataset["tags"].apply(lambda x:x.lower())

In [21]:
import nltk 
import re # remove , . ; -

In [22]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [23]:
def stem(text):
    y=[]
    text = re.sub(r"[^a-zA-Z\s]","",text)
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [24]:
dataset["tags"] = dataset["tags"].apply(stem)

In [25]:
from sklearn.feature_extraction.text import CountVectorizer

In [26]:
cv = CountVectorizer(max_features=5000,stop_words="english")

In [27]:
vector = cv.fit_transform(dataset["tags"]).toarray()

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
simi = cosine_similarity(vector)

In [30]:
def reco(movie):
    index = dataset[dataset["title"] == movie].index[0]
    distance = simi[index]
    movie_list = sorted(list(enumerate(distance)),reverse=True,key=lambda x:x[1])[1:11]

    for i in movie_list:
        print(dataset.iloc[i[0]].title)

In [31]:
reco("Avatar")

Aliens vs Predator: Requiem
Independence Day
Titan A.E.
Aliens
Lifeforce
Ender's Game
Meet Dave
Attack the Block
The Host
The Host


In [32]:
import pickle
pickle.dump(dataset,open("movie predicter.pkl","wb"))

In [33]:
pickle.dump(simi,open("similarity.pkl","wb"))

In [32]:
dataset.head()

,title,tags,id
0,Avatar,in the nd centuri a parapleg marin is dispatch...,19995
1,Pirates of the Caribbean: At World's End,captain barbossa long believ to be dead ha com...,285
2,Spectre,a cryptic messag from bond past send him on a ...,206647
3,The Dark Knight Rises,follow the death of district attorney harvey d...,49026
4,John Carter,john carter is a warweari former militari capt...,49529
